# Implementation on the GPU-based system

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$This notebook explains how the GPU timing estimates for the classical MCMC algorithm are obtained.

**Table of contents**
1. Source
2. Algorithm
3. How to estimate the timing
4. Reproducibility

## Source

Both the code and the data are available in the `timing_estimation/gpu` folder.

The folder contains:

* `timing_estimation_gpu.cu`: CUDA/C++ source file containing the SK model generation, GPU memory wrappers, random-number pools, chain-state storage, local and uniform Metropolis kernels, and CUDA-event timing logic.
* `timing_estimation_gpu.slurm`: SLURM script that compiles `timing_estimation_gpu.cu` for different values of `SK_N` and runs the resulting GPU benchmarks. These are generic settings; we used slightly different ones, as described below.
* `h100/test<i>`: folders containing identical runs executed at different times on our device. Each folder contains a copy of the source code, the `*.slurm` file used for execution, and the log files reporting the hardware characteristics, compiler configuration, and timing results.

## Algorithm

The benchmark compares the local and uniform proposals. The local proposal computes $\Delta_i E = 2s_i(h_i+\sum_j J_{ij}s_j)$, requiring $O(N)$ arithmetic per proposal. The uniform proposal samples a new spin configuration and computes its full dense SK energy, requiring $O(N^2)$ arithmetic per proposal. The GPU implementation uses one CUDA block per Markov chain, with `BLOCK_THREADS = 128` threads per block. The chain evolution is still sequential in Monte Carlo time, so the relevant GPU parallelism comes from parallelizing the arithmetic inside each proposal. Ideally, the arithmetic time for one proposal scales like $O(N/T)$ and $O(N^2/T)$ for local and uniform moves respectively, where $T$ is the number of threads per CUDA block, up to synchronization and reduction overheads.

### GPU common utilities

In CUDA, a `__global__` function is a kernel: it is launched from the CPU and executed on the GPU by many threads. A `__shared__` variable lives in shared memory local to one CUDA block: all threads in the same block can access it, while different blocks have independent copies. In this benchmark, shared memory is used for block-level reductions, for the trial state in the uniform move, and for small scalar variables shared by the threads evolving one chain.

**Constants**

* `SRC`: CUDA/C++ source file compiled by the SLURM script.
* `NS`: list of system sizes, here $N\in\{64,128,256,512\}$.
* `SK_N`: number of spins, passed at compile time with `-DSK_N=<N>` for each value in `NS`.
* `N_MODELS`: number of independent SK instances generated inside one benchmark execution.
* `CHAINS_PER_MODEL`: number of chains simulated for each SK instance.
* `N_STEPS`: number of Metropolis steps per chain.
* `SEED`: seed used for model generation and random-pool generation.
* `BETA`: inverse temperature used in the Metropolis probability.
* `BLOCK_THREADS`: number of CUDA threads assigned to one chain/block.
* `RNG_POOL_SIZE`: size of the precomputed random-number pools.
* `GPU_ARCH`: CUDA architecture passed to `nvcc`. In the attached SLURM script this is `sm_80`; the run was executed on an NVIDIA H100 NVL GPU, as shown in the attached log.
* `N_CHAINS`: total number of chains, defined as `N_MODELS * CHAINS_PER_MODEL`.

**Helper classes and methods**

* `CUDA_CHECK`: macro wrapping CUDA runtime calls. If a CUDA call fails, it prints the CUDA error string and terminates the program. This is used after memory operations, kernel launches, and synchronization calls.
* `DeviceArray<T>`: wrapper for GPU memory. It owns a device pointer allocated with `cudaMalloc` and releases it with `cudaFree` in the destructor. Copying is disabled to avoid double ownership of the same GPU pointer. The methods `data()` and `data() const` expose the raw device pointer, while `copy_from(v)` copies a host `std::vector<T>` to the device allocation.
* `GpuTimer`: CUDA-event timer used to measure kernel runtime. The constructor creates two CUDA events, `start()` records the first event, and `stop_seconds()` records the second event, synchronizes on it, computes the elapsed time in milliseconds with `cudaEventElapsedTime`, and returns the result in seconds. This keeps the timing restricted to the GPU kernel region.
* `RandomPool`: host-side owner of the random-number pools. It generates two arrays before the timed region: `h_int`, containing random `uint64_t` integers, and `h_float`, containing uniform floats in $[0,1)$. These arrays are copied to the GPU into `d_int` and `d_float`. This is done so that the timed kernels do not include the cost of RNG generation.
* `RandomPoolView`: device-side view of the random-number pools. It stores only device pointers and provides `__device__` methods usable inside kernels. The method `randint(chain, step, tag)` returns a precomputed integer, `uniform(chain, step, tag)` returns a precomputed uniform float, and `spin(chain, step, i)` extracts one random spin in $\{-1,+1\}$ from an integer word. The pool is indexed using a deterministic combination of chain id, MCMC step, and tag, and is then reduced modulo `RNG_POOL_SIZE`. Therefore random numbers are generated once and recycled cyclically during the timed kernel. This is intentional: the goal is to benchmark the arithmetic and memory behavior of the Metropolis kernels, not the cost of GPU random-number generation. This would not be appropriate in a production sampler unless the same reuse scheme were statistically justified or the random stream were supplied externally.
* `ModelBank`: host-side owner of the SK instances. It stores all fields in `h_host` and all dense coupling matrices in `J_host`, generates normalized SK instances in `init_models(seed)`, and copies them to GPU arrays `h_dev` and `J_dev`. The method `energy(model, s)` computes the dense SK energy on the host for a given spin state and is used to initialize the chain energies.
* `ModelBankView`: device-side view of the model bank. The method `h_model(model)` returns a pointer to the field vector of one model, while `J_model(model)` returns a pointer to the dense $N\times N$ coupling matrix of that model. Kernels use this view to access the coefficients assigned to each chain.
* `ChainState`: host-side owner of the device chain state. It allocates `s_dev`, storing all spin configurations, and `E_dev`, storing one current energy per chain. It also computes `E0_host`, the all-plus initial energy for each chain, using the corresponding SK model. The method `reset_energy()` copies these initial energies back to the GPU before timing each kernel.
* `ChainStateView`: device-side view of the chain state. The method `state(chain)` returns the spin array for one chain, `energy(chain)` returns a reference to the current energy of that chain, `flip(chain, i)` flips one spin, and `reset_state(chain)` resets a chain to the all-plus state using all threads in the block.
* `IsingEnergyBuffer`: per-block shared-memory scratch object. It contains `red[BLOCK_THREADS]`, used for reductions, and `trial[SK_N]`, used by the uniform proposal to store the proposed spin configuration. Its method `block_sum(x)` performs a binary-tree shared-memory reduction over the block. The method `local_delta_energy(h,J,s,i)` computes $\Delta_i E$ by splitting the row sum over block threads and reducing the partial sums. The method `assign_random_state_to_trial(rng,chain,step)` fills the shared trial state with random spins. The method `get_energy_trial_state(h,J)` computes the dense SK energy of the trial state. The method `accept_trial_state(s)` copies the accepted trial state into the persistent chain state.
* `metropolis_accept`: device-side Metropolis rule. It accepts automatically if $dE\leq 0$, otherwise it compares a precomputed uniform random number with `__expf(-BETA * dE)`. The use of `__expf` is consistent with compiling with `--use_fast_math`.
* `reset_states`: `__global__` kernel that resets all chains to the all-plus spin configuration on the GPU. It is called before timing either proposal kernel and is outside the measured proposal-kernel region.

### GPU local move implementation

The local move is implemented by the `local_kernel` CUDA kernel. The kernel is launched with one block per chain and `BLOCK_THREADS` threads per block. Thus `blockIdx.x` selects the Markov chain, and `threadIdx.x` selects the worker thread inside that chain.

For each MCMC step, thread `0` samples the spin index $i$ using `rng.randint`. The index is stored in `__shared__ int i`, so all threads in the block can use it after `__syncthreads()`. The block then calls `energy_helper.local_delta_energy(h,J,s,i)`, where each thread accumulates part of the row sum $\sum_j J_{ij}s_j$. The partial sums are combined by the shared-memory reduction `block_sum`.

Only thread `0` performs the Metropolis accept/reject test. If the move is accepted, it flips spin $i$ in the persistent chain state and updates the stored energy by adding $dE$. A synchronization at the end of the step ensures that all threads see the updated state before the next proposal.

### GPU uniform move implementation

The uniform move is implemented by the `uniform_kernel` CUDA kernel. Again, one CUDA block evolves one Markov chain.

At each MCMC step, all threads first call `assign_random_state_to_trial`, which fills the shared-memory array `trial[SK_N]` with a completely new spin configuration. The proposed state is kept in shared memory until the Metropolis decision is known. The block then calls `get_energy_trial_state(h,J)`. This distributes the dense energy computation $-\sum_i h_i s_i-\sum_{i<j}J_{ij}s_i s_j$ across the block threads. Each thread accumulates part of the field and interaction terms, and the partial sums are combined through `block_sum`.

Thread `0` computes $dE=E_{\mathrm{new}}-E_{\mathrm{current}}$ and applies the Metropolis rule. If the move is accepted, it updates the stored energy, and all threads cooperatively copy the shared trial state into the persistent chain state through `accept_trial_state`.


## How to estimate the timing

Timing is measured with CUDA events around the proposal kernels only. Memory allocation, host-to-device transfers, random-pool construction, state initialization, and model generation are outside the timed region.

The script compiles `SRC = timing_estimation_gpu.cu` for $N\in\{64,128,256,512\}$ using `-DSK_N=<N>`. In the attached script, the compile-time parameters are `N_MODELS = 1`, `CHAINS_PER_MODEL = 1`, `N_STEPS = 1000000`, `SEED = 123456789`, `BETA = 1.0f`, `BLOCK_THREADS = 128`, `RNG_POOL_SIZE = 1048576`, and `GPU_ARCH = sm_80`. The attached log shows that the run was executed on an NVIDIA H100 NVL GPU.

Since `N_MODELS = 1` and `CHAINS_PER_MODEL = 1`, we have `N_CHAINS = 1`. Therefore the reported kernel times correspond to a single-chain latency test. They should not be interpreted as saturated H100 throughput numbers. A many-chain throughput benchmark would require increasing `N_MODELS`, `CHAINS_PER_MODEL`, or both, so that many CUDA blocks are resident across the GPU.

The code reports `local_seconds` and `uniform_seconds`. For the current single-chain configuration, the average time per attempted proposal is `seconds / N_STEPS`. In a many-chain configuration, it would be `seconds / (N_CHAINS * N_STEPS)`.

The table below contains three timing runs with the same benchmark configuration. The attached log file reproduces Run 1 exactly; the other two rows are retained as additional runs from the same experiment set.

The timing of the local moves in seconds is the CUDA-event kernel time for one full run of `N_STEPS = 1000000` attempted proposals:

| Run | N=64 | N=128 | N=256 | N=512 |
|---|---:|---:|---:|---:|
| Run 1   | 0.928508 | 0.937455 | 1.125960 | 1.557990 |
| Run 2   | 0.926027 | 0.935282 | 1.121310 | 1.547510 |
| Run 3   | 0.926223 | 0.935248 | 1.117720 | 1.547070 |
| Average | 0.926919 | 0.935995 | 1.121663 | 1.550857 |

The timing of the uniform moves in seconds is the CUDA-event kernel time for one full run of `N_STEPS = 1000000` attempted proposals:

| Run | N=64 | N=128 | N=256 | N=512 |
|---|---:|---:|---:|---:|
| Run 1   | 2.050120 | 3.196240 | 8.695600 | 59.846200 |
| Run 2   | 2.051920 | 3.201940 | 8.702960 | 59.349300 |
| Run 3   | 2.051810 | 3.201820 | 8.697330 | 59.360100 |
| Average | 2.051283 | 3.200000 | 8.698630 | 59.518533 |


In [1]:
import numpy as np
from scipy.optimize import nnls

N_OPS = 1_000_000
ns = np.array([64, 128, 256, 512.])

# GPU timings: rows = independent runs, columns = N = 64, 128, 256, 512.
local_time_gpu = np.array([
    [0.928508, 0.937455, 1.125960, 1.557990],
    [0.926027, 0.935282, 1.121310, 1.547510],
    [0.926223, 0.935248, 1.117720, 1.547070],
])

uniform_time_gpu = np.array([
    [2.050120, 3.196240, 8.695600, 59.846200],
    [2.051920, 3.201940, 8.702960, 59.349300],
    [2.051810, 3.201820, 8.697330, 59.360100],
])

avg_local_time_gpu = np.mean(local_time_gpu, axis=0)
avg_uniform_time_gpu = np.mean(uniform_time_gpu, axis=0)

def fit(y, deg):
    X = np.vstack([ns**k for k in range(deg + 1)]).T
    return nnls(X, y)[0]

a, b = fit(avg_local_time_gpu, 1)
A, B, C = fit(avg_uniform_time_gpu, 2)

print("Compilation settings: gpu_h100_nvl_sm80")
print(f"  avg local timings:   {avg_local_time_gpu}")
print(f"  avg uniform timings: {avg_uniform_time_gpu}")
print(f"  local:      {a:.3e} + {b:.3e} n")
print(f"  uniform:    {A:.3e} + {B:.3e} n + {C:.3e} n^2")
print(f"  local/op:   {a/N_OPS:.3e} + {b/N_OPS:.3e} n")
print(f"  uniform/op: {A/N_OPS:.3e} + {B/N_OPS:.3e} n + {C/N_OPS:.3e} n^2")


Compilation settings: gpu_h100_nvl_sm80
  avg local timings:   [0.92691933 0.935995   1.12166333 1.55085667]
  avg uniform timings: [ 2.05128333  3.2         8.69863    59.51853333]
  local:      7.837e-01 + 1.459e-03 n
  uniform:    0.000e+00 + 0.000e+00 n + 2.215e-04 n^2
  local/op:   7.837e-07 + 1.459e-09 n
  uniform/op: 0.000e+00 + 0.000e+00 n + 2.215e-10 n^2


## Reproducibility

The run is controlled by `timing_estimation_gpu.slurm`. The script in the main folder loads GCC 12.2.0 and tries to load CUDA 12.1, falling back to the default CUDA module if CUDA 12.1 is unavailable. In the attached log in the `h100` folder, a different system from the Cineca cluster is used and that loads CUDA compilation tools 13.1.115. We have run the program on a system with NVIDIA H100 NVL GPU.

The attached script compiles with `GPU_ARCH = sm_80`, so the main compilation flags are `-std=c++17 -O3 --use_fast_math -arch=sm_80 -lineinfo -Xcompiler=-O3 -Xcompiler=-DNDEBUG`. The Metropolis exponential uses fast single-precision math through `__expf` (`--use_fast_math`).
